<a href="https://colab.research.google.com/github/the-blackunicorn/MSDSM-Portfolio/blob/main/Internships/DVPL-RAG-Chatbot/ChatBot-DVPL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install required libraries
!pip install langchain chromadb sentence-transformers -q

In [ ]:
!pip install langchain-text-splitters -q

In [ ]:
import re

def parse_whatsapp(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    messages = []
    pattern = re.compile(r'\d+/\d+/\d+,\s+\d+:\d+\u202f[AP]M - ([^:]+): (.+)')

    for line in lines:
        match = pattern.match(line)
        if match:
            sender, message = match.groups()
            if '<Media omitted>' in message: continue
            if 'This message was deleted' in message: continue
            if len(message.strip()) < 5: continue
            messages.append(f"{sender.strip()}: {message.strip()}")

    return messages

msgs1 = parse_whatsapp('/content/ChatBot-DVPL/WhatsApp Chat with NRDA Contractor.txt')
msgs2 = parse_whatsapp('/content/ChatBot-DVPL/WhatsApp Chat with Nrda Aayog site.txt')

all_msgs = msgs1 + msgs2

print(f"File 1: {len(msgs1)} messages")
print(f"File 2: {len(msgs2)} messages")
print(f"Total: {len(all_msgs)} messages")
print("\nSample:")
for m in all_msgs[:5]:
    print(m)

File 1: 37 messages
File 2: 4616 messages
Total: 4653 messages

Sample:
!! जय श्री महाकाल !!: Sabhi thekedar apna labour supply 1-12-25 se 15-12-25 tak ka duplicate voucher ki copy account office me jama kre..
Rk Sahu Sir Dv: सरिया शटरिंग वाले सभी ठेकेदार बेंडिंग मशीन को बंद करने के बाद उसके पिन को निकाल कर उसमें आयल डालकर साफ कर फिर  चालू करें
+91 90061 11835: G sir
Er. Abhay Sir Dv: @⁨vidhya Ele..⁩
Er. Abhay Sir Dv: Check kar lo


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Join all messages into one big text
full_text = "\n".join(all_msgs)

# Split into chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_text(full_text)

print(f"Total chunks: {len(chunks)}")
print("\nSample chunk:")
print(chunks[0])

Total chunks: 475

Sample chunk:
!! जय श्री महाकाल !!: Sabhi thekedar apna labour supply 1-12-25 se 15-12-25 tak ka duplicate voucher ki copy account office me jama kre..
Rk Sahu Sir Dv: सरिया शटरिंग वाले सभी ठेकेदार बेंडिंग मशीन को बंद करने के बाद उसके पिन को निकाल कर उसमें आयल डालकर साफ कर फिर  चालू करें
+91 90061 11835: G sir
Er. Abhay Sir Dv: @⁨vidhya Ele..⁩
Er. Abhay Sir Dv: Check kar lo
Rk Sahu Sir Dv: सभी ठेकेदार अकाउंट में ऑफिस में आकर खर्ची ले ले
!! जय श्री महाकाल !!: 12 bje contractor payment hooga


In [ ]:
from sentence_transformers import SentenceTransformer
import chromadb

# Better multilingual model
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

# Fresh ChromaDB
client = chromadb.Client()
collection = client.get_or_create_collection("whatsapp_chats_v2")

# Embed and store
print("Embedding chunks... please wait")
embeddings = model.encode(chunks, show_progress_bar=True)

collection.add(
    documents=chunks,
    embeddings=embeddings.tolist(),
    ids=[str(i) for i in range(len(chunks))]
)

print(f"\nDone! {collection.count()} chunks stored in DB")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embedding chunks... please wait


Batches:   0%|          | 0/15 [00:00<?, ?it/s]


Done! 475 chunks stored in DB


In [ ]:
def search(query, n_results=3):
    query_embedding = model.encode([query]).tolist()
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=n_results
    )
    print(f"Query: {query}\n")
    for i, doc in enumerate(results['documents'][0]):
        print(f"--- Result {i+1} ---")
        print(doc)
        print()

# Test with a sample query
search("contractor kharchi kab milega")

Query: contractor kharchi kab milega

--- Result 1 ---
Rk Sahu Sir Dv: Kal texture wala site aayega
Rk Sahu Sir Dv: @⁨Arun Banjare Plumber⁩ @⁨vidhya Ele..⁩ Kal morning 7 baje dewatering pump start kar sump and lift pit ka pani Khali karaye
Arun Banjare Plumber: Ok sir
Rk Sahu Sir Dv: @⁨Raju Anna Machanical⁩ please line up tower crane foundation Bolt and other requirements and Verma ji
Rk Sahu Sir Dv: Foundation steel are ready
Sanjay Vishwakarma Ji Dv: Batching plant
Narayan Jaiswal Ji Dv: 45 ka self screw chahiye

--- Result 2 ---
Surendra Supervisor Dv: Propjuck. 5 MTR 910 pic
Rk Sahu Sir Dv: @⁨vidhya Ele..⁩ DG ka kya hua
Raju Anna Machanical: Abhi tak Koi nahin aaya  DG ka pass Mera mechanical staf kada hi jab jaruri hi tho start kardeyaga
Aanand Pandey Dv: aa gaye ha ghumke ana hota ha RTO ke wajah se
Er. Abhay Sir Dv: @⁨Raju Anna Machanical⁩ IPS flooring ke liye SKPL concrete 3 cum.aana tha mukku bhaiya se bat karke bhejwa do.
Raju Anna Machanical: Ok sir tm nikal rahahu

--- Resu

In [ ]:
!pip install groq -q

In [ ]:
from groq import Groq

# Paste your Groq API key here
groq_client = Groq(api_key=userdata.get('GROQ_API_KEY'))

def ask(query):
    # Step 1: Retrieve relevant chunks
    query_embedding = model.encode([query]).tolist()
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=3
    )
    context = "\n\n".join(results['documents'][0])

    # Step 2: Send to LLM
    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "system",
                "content": "You are a helpful assistant. Answer the user's question based only on the provided WhatsApp chat context. The chats are from a construction site in India, mixed Hindi and English. If the answer is not in the context, say 'I don't have information about this.'"
            },
            {
                "role": "user",
                "content": f"Context:\n{context}\n\nQuestion: {query}"
            }
        ]
    )
    print(f"Answer: {response.choices[0].message.content}")

# Test
ask("contractor kharchi kab milega")

Answer: 3:00 ke baad, account room mein Milega.


In [ ]:
ask("labour ID card kab banega")
ask("bending machine ke baare mein kya instructions hain")
ask("driver ka phone number kya hai")

Answer: आपको ऑफिस के सामने जहां लेबर खाना खाते हैं, वहां जाना होगा, वहीं पर लेबर का आईडी कार्ड बनवाने के लिए सूचना दी जा रही है। समय के बारे में विशिष्ट जानकारी नहीं है, लेकिन सभी ठेकेदारों को सूचित किया जा रहा है।
Answer: Raju Anna Machanical ko instructions diya gaya hai ki vidhya aur 1 bending machine abi pahuncha hai, uska vuskabi (vasculab/test) dekhna hai.
Answer: Yeswant ka phone number 6264409717 hai, Ses Narayan driver ka phone number 8817294998 hai, aur Gulsan driver ka phone number 6268552475 hai.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pickle

# Save chunks
with open('/content/drive/MyDrive/chatbot_chunks.pkl', 'wb') as f:
    pickle.dump(chunks, f)

# Save embeddings
import numpy as np
embeddings_array = model.encode(chunks)
np.save('/content/drive/MyDrive/chatbot_embeddings.npy', embeddings_array)

print("Saved successfully!")

Saved successfully!


In [ ]:
# Load chunks and embeddings from Drive
with open('/content/drive/MyDrive/chatbot_chunks.pkl', 'rb') as f:
    loaded_chunks = pickle.load(f)

loaded_embeddings = np.load('/content/drive/MyDrive/chatbot_embeddings.npy')

print(f"Chunks loaded: {len(loaded_chunks)}")
print(f"Embeddings shape: {loaded_embeddings.shape}")

Chunks loaded: 475
Embeddings shape: (475, 384)


In [ ]:
!pip install streamlit pyngrok -q

In [ ]:
%%writefile app.py

import streamlit as st
from sentence_transformers import SentenceTransformer
from groq import Groq
import chromadb
import pickle
import numpy as np

st.title("🏗️ NRDA Site Chatbot")
st.caption("Ask anything from the WhatsApp chats")

@st.cache_resource
def load_everything():
    model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
    with open('/content/drive/MyDrive/chatbot_chunks.pkl', 'rb') as f:
        chunks = pickle.load(f)
    embeddings = np.load('/content/drive/MyDrive/chatbot_embeddings.npy')
    client = chromadb.Client()
    collection = client.get_or_create_collection("whatsapp_streamlit")
    collection.add(
        documents=chunks,
        embeddings=embeddings.tolist(),
        ids=[str(i) for i in range(len(chunks))]
    )
    return model, collection, chunks

model, collection, chunks = load_everything()
groq_client = Groq(api_key=userdata.get('GROQ_API_KEY'))

if "messages" not in st.session_state:
    st.session_state.messages = []

for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.write(msg["content"])

if query := st.chat_input("Kuch bhi pucho..."):
    st.session_state.messages.append({"role": "user", "content": query})
    with st.chat_message("user"):
        st.write(query)

    # Semantic search
    query_embedding = model.encode([query]).tolist()
    results = collection.query(query_embeddings=query_embedding, n_results=6)
    context = "\n\n".join(results['documents'][0])

    # Month-aware keyword fallback
    month_map = {
        "january": "/01/", "february": "/02/", "march": "/03/", "april": "/04/",
        "may": "/05/", "june": "/06/", "july": "/07/", "august": "/08/",
        "september": "/09/", "october": "/10/", "november": "/11/", "december": "/12/"
    }

    query_lower = query.lower()
    keyword_matches = []

    # Month detection first
    for month, num in month_map.items():
        if month in query_lower:
            all_month_chunks = [c for c in chunks if num in c]
            # Filter TM-heavy chunks
            filtered = [c for c in all_month_chunks if c.count("Tm no.") < 3 and len(c) > 150]
            # Pick chunks from different parts (start, middle, end)
            if len(filtered) > 6:
                step = len(filtered) // 6
                keyword_matches = [filtered[i*step] for i in range(6)]
            else:
                keyword_matches = filtered
            break

    # If no month found, regular keyword search
    if not keyword_matches:
        words = [w for w in query_lower.split() if len(w) > 4]
        keyword_matches = [c for c in chunks if any(w in c.lower() for w in words)]

    if keyword_matches:
        context = "\n\n".join(keyword_matches[:6]) + "\n\n" + context

    # Build history
    history = [{"role": "system", "content": "You are a helpful assistant for a construction site in India. Answer ONLY using exact information present in the context. Do NOT add assumptions or general knowledge. If a number appears in context, mention exactly who it belongs to as stated. For broad time-period questions, give a concise 3-4 line summary, not a bullet list. If the context is incomplete, add 'Poori jaankari available nahi hai.' Always reply in Hinglish - Hindi words written in English letters, use Hindi script only when asked in hindi."}]

    for msg in st.session_state.messages[-6:]:
        history.append({"role": msg["role"], "content": msg["content"]})

    history.append({"role": "user", "content": f"Context:\n{context}\n\nQuestion: {query}"})

    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=history
    )

    answer = response.choices[0].message.content
    st.session_state.messages.append({"role": "assistant", "content": answer})
    with st.chat_message("assistant"):
        st.write(answer)

Overwriting app.py


In [ ]:
from pyngrok import ngrok
import subprocess
import time

# Kill all existing tunnels
ngrok.kill()
time.sleep(2)

# Restart streamlit
subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501"])
time.sleep(3)

# Fresh tunnel
public_url = ngrok.connect(8501)
print(f"Open this link: {public_url}")

Open this link: NgrokTunnel: "https://jiffy-thinning-wieldable.ngrok-free.dev" -> "http://localhost:8501"


In [ ]:
from google.colab import files
files.download('/content/drive/MyDrive/chatbot_chunks.pkl')
files.download('/content/drive/MyDrive/chatbot_embeddings.npy')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>